In [1]:
import logging
import json
import numpy as np
import inspect


from pathlib import Path
from typing import Optional, Dict, Any
from utils.paths import get_paths
from utils.file_io import save_data
from utils.logging_setup import configure_logging, log_layer_paths
from utils.pipeline_config_loader import (
    load_pipeline_config,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.truths import (
    make_process_run_id,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record_by_hash,
    get_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.synthetic_profiles import (
    load_rich_profile_csv,
    load_correlation_pairs_csv,
    load_group_map_csv,
    load_fault_pairings_csv,
)
from utils.synthetic_generator import SyntheticGenerator, EpisodeSpec

from utils.synthetic_postgres_writer import (
    get_engine,
    ensure_sequence,
    reserve_next_batch_id,
    reserve_cycle_range,
    write_stream_batch,
)

from utils.synthetic_export import export_synthetic_batch_to_parquet
from utils.pipeline_config_loader import build_truth_config_block


In [2]:
# --- Notebook params ---
STAGE = "synthetic"
DATASET = "pump"
MODE = "train"
PROFILE = "default"

In [ ]:
paths = get_paths()

config_obj = load_pipeline_config(
    config_root=paths.configs,
    stage=STAGE,
    dataset=DATASET,
    mode=MODE,
    profile=PROFILE,
    project_root=paths.root,
)
CONFIG = config_obj.data

SYN_CFG = CONFIG["synthetic"]
PATHS = CONFIG["resolved_paths"]
PIPELINE = CONFIG.get("pipeline", {"execution_mode": "batch", "orchestration_mode": "notebook"})

PIPELINE_MODE = PIPELINE["execution_mode"]
DATASET_NAME = str(CONFIG["dataset"]["name"]).strip().lower()
TRUTH_VERSION = CONFIG["versions"]["truth"]

TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])
LOGS_PATH = Path(PATHS["logs_root"])
ARTIFACTS_ROOT = Path(PATHS["artifacts_root"])


TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)



set_wandb_dir_from_config(CONFIG)

print("DATASET_NAME:", DATASET_NAME)
print("TRUTHS_PATH:", TRUTHS_PATH)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)

DATASET_NAME: pump
TRUTHS_PATH: /workspace/artifacts/truths
ARTIFACTS_ROOT: /workspace/artifacts


In [4]:
# Logging Setup

# Create gold log path 
synthetic_log_path = paths.logs / "synthetic_data_generator.log"

# Initial Logger
configure_logging(
    "capstone",
    synthetic_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.synthetic")

# Log load and initiation
logger.info("Synethetic Data Generation starting")

# Log paths loads
log_layer_paths(paths, current_layer="synthetic", logger=logger)


2026-03-14 01:39:52,099 | INFO | capstone.synthetic | Synethetic Data Generation starting
2026-03-14 01:39:52,101 | INFO | capstone.synthetic | Project Root Path Loaded: /workspace
2026-03-14 01:39:52,103 | INFO | capstone.synthetic | Project Logging Path Loaded: /workspace/logs
2026-03-14 01:39:52,105 | INFO | capstone.synthetic | Project Artifacts Path Loaded: /workspace/artifacts
2026-03-14 01:39:52,108 | INFO | capstone.synthetic | Project Notebooks Path Loaded: /workspace/notebooks
2026-03-14 01:39:52,110 | INFO | capstone.synthetic | Project Truths Path Loaded: /workspace/artifacts/truths
2026-03-14 01:39:52,112 | INFO | capstone.synthetic | Project Data Path Loaded: /workspace/data
2026-03-14 01:39:52,115 | INFO | capstone.synthetic | Previous Layer (Gold) Path Loaded: /workspace/data/gold


In [5]:
def sample_int_range(rng: np.random.Generator, value_or_range, *, low_inclusive: bool = True) -> int:
    """
    Accepts either:
      - int
      - [low, high]
    Returns an int.
    """
    if isinstance(value_or_range, (int, np.integer)):
        return int(value_or_range)

    if isinstance(value_or_range, (list, tuple)) and len(value_or_range) == 2:
        low = int(value_or_range[0])
        high = int(value_or_range[1])
        if low_inclusive:
            # numpy integers are low inclusive, high exclusive
            return int(rng.integers(low, high + 1))
        return int(rng.integers(low, high))

    raise TypeError(f"Expected int or [low, high] range, got: {type(value_or_range)} -> {value_or_range}")


def sample_float_range(rng: np.random.Generator, value_or_range) -> float:
    """
    Accepts either:
      - float/int
      - [low, high]
    Returns a float.
    """
    if isinstance(value_or_range, (float, int, np.floating, np.integer)):
        return float(value_or_range)

    if isinstance(value_or_range, (list, tuple)) and len(value_or_range) == 2:
        low = float(value_or_range[0])
        high = float(value_or_range[1])
        return float(rng.uniform(low, high))

    raise TypeError(f"Expected float or [low, high] range, got: {type(value_or_range)} -> {value_or_range}")

In [6]:
# Get Latest Truth Hash

def get_latest_truth_hash(
    *,
    truth_index_path: Path,
    layer_name: str,
    dataset_name: str,
) -> str:
    """
    Return the most recent truth_hash for a given layer + dataset from truth_index.jsonl.

    Assumes truth_index.jsonl is append-only and newer entries are later in the file.
    """
    if not truth_index_path.exists():
        raise FileNotFoundError(f"truth_index.jsonl not found: {truth_index_path}")

    dataset_name_norm = str(dataset_name).strip().lower()
    layer_name_norm = str(layer_name).strip().lower()

    latest_record: Optional[Dict[str, Any]] = None

    with truth_index_path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            rec_layer = str(rec.get("layer_name", "")).strip().lower()
            rec_dataset = str(rec.get("dataset_name", "")).strip().lower()

            if rec_layer == layer_name_norm and rec_dataset == dataset_name_norm:
                latest_record = rec

    if latest_record is None:
        raise ValueError(
            f"No truth records found for layer='{layer_name}' dataset='{dataset_name}' in {truth_index_path}"
        )

    truth_hash = latest_record.get("truth_hash")
    if truth_hash is None or str(truth_hash).strip() == "":
        raise ValueError("Latest record is missing a usable truth_hash.")

    return str(truth_hash).strip()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


def get_latest_silver_eda_truth_hash(
    *,
    truth_index_path: Path,
    dataset_name: str,
) -> str:
    """
    Convenience wrapper: latest Silver EDA truth hash for a dataset.
    """
    return get_latest_truth_hash(
        truth_index_path=truth_index_path,
        layer_name="silver_eda",
        dataset_name=dataset_name,
    )

In [7]:
# Updated
# --- Notebook params ---
STAGE = SYN_CFG["layer_name"]
DATASET = "pump"
MODE = "train"
PROFILE = "default"

# Parent truth hash from your latest Silver EDA run
SILVER_EDA_TRUTH_HASH = get_latest_silver_eda_truth_hash(truth_index_path=TRUTH_INDEX_PATH, dataset_name=DATASET_NAME,)
print("Latest SILVER_EDA_TRUTH_HASH:", SILVER_EDA_TRUTH_HASH)

# Faults
# Episode overrides (easy test knobs)
PRIMARY_SENSOR = None          # None => first sensor
PRIMARY_FAULT_TYPE = list(SYN_CFG["faults"]["allowed"])


# Episode Settings
NORMAL_BEFORE = list(SYN_CFG["episode"]["normal_before_range"])
BUILDUP = list(SYN_CFG["episode"]["buildup_range"])
FAILURE = list(SYN_CFG["episode"]["failure_range"])
RECOVERY = list(SYN_CFG["episode"]["recovery_range"])
NORMAL_AFTER = list(SYN_CFG["episode"]["normal_after_range"])
MAGNITUDE = list(SYN_CFG["episode"]["magnitude_range"])

SYNTH_PROCESS_RUN_ID = make_process_run_id(SYN_CFG["process_run_id_prefix"])

# Outputs
OUTPUT_MODE = SYN_CFG["output_mode"]

# Postgres settings
PG_SCHEMA = str(SYN_CFG["postgres"]["schema"])
TABLE_ARTIFACT_NAME = str(SYN_CFG["postgres"]["table_artifact_name"])

# medallion naming: synthetic_<dataset>_<artifact_name>
ARTIFACT_NAME = "stream"       

# Export
EXPORT_ENABLED = bool(SYN_CFG["export"]["enabled"])
EXPORT_DIRECTORY = str(SYN_CFG["export"]["export_dir_key"])

Latest SILVER_EDA_TRUTH_HASH: 72f16602506edf152a82982409835f2e6ec1039928a79e202fac88a5b69e5de4


In [8]:
if SILVER_EDA_TRUTH_HASH is None or str(SILVER_EDA_TRUTH_HASH).strip() == "":
    raise ValueError("Set SILVER_EDA_TRUTH_HASH in the parameter cell.")

silver_eda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver_eda",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_EDA_TRUTH_HASH).strip(),
)

PARENT_TRUTH_HASH = get_truth_hash(silver_eda_truth)

parent_mode = get_pipeline_mode_from_truth(silver_eda_truth)
if parent_mode:
    PIPELINE_MODE = parent_mode

print("PARENT_TRUTH_HASH:", PARENT_TRUTH_HASH)
print("PIPELINE_MODE:", PIPELINE_MODE)

logger.info("W&B PARENT_TRUTH_HASH: %s", PARENT_TRUTH_HASH)
logger.info("W&B PIPELINE_MODE: %s", PIPELINE_MODE)

2026-03-14 01:39:52,229 | INFO | capstone.synthetic | W&B PARENT_TRUTH_HASH: 72f16602506edf152a82982409835f2e6ec1039928a79e202fac88a5b69e5de4
2026-03-14 01:39:52,231 | INFO | capstone.synthetic | W&B PIPELINE_MODE: batch


PARENT_TRUTH_HASH: 72f16602506edf152a82982409835f2e6ec1039928a79e202fac88a5b69e5de4
PIPELINE_MODE: batch


In [9]:
keys = SYN_CFG["silver_eda_artifact_keys"]

profile_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_normal"])
profile_abnormal_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_abnormal"])
profile_recovery_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_recovery"])

corr_pairs_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["corr_pairs_normal"])
group_map_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["group_map_normal"])
fault_pairings_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["fault_pairings_normal"])

print(profile_normal_path)
print(profile_abnormal_path)
print(profile_recovery_path)
print(corr_pairs_normal_path)
print(group_map_normal_path)
print(fault_pairings_normal_path)

logger.info("silver_eda_artifact_keys: %s", keys)


2026-03-14 01:39:52,248 | INFO | capstone.synthetic | silver_eda_artifact_keys: {'profile_normal': 'feature_profile_normal_path', 'profile_abnormal': 'feature_profile_abnormal_path', 'profile_recovery': 'feature_profile_recovery_path', 'corr_pairs_normal': 'sensor_correlation_pairs_normal_path', 'group_map_normal': 'sensor_group_map_normal_path', 'fault_pairings_normal': 'sensor_fault_pairings_normal_path'}


/workspace/artifacts/silver_eda/pump/feature_profile_normal.csv
/workspace/artifacts/silver_eda/pump/feature_profile_abnormal.csv
/workspace/artifacts/silver_eda/pump/feature_profile_recovery.csv
/workspace/artifacts/silver_eda/pump/sensor_correlation_pairs_normal.csv
/workspace/artifacts/silver_eda/pump/sensor_group_map_normal.csv
/workspace/artifacts/silver_eda/pump/sensor_fault_pairings_normal.csv


In [10]:
normal_profiles = load_rich_profile_csv(profile_normal_path, state_scope="normal")
abnormal_profiles = load_rich_profile_csv(profile_abnormal_path, state_scope="abnormal")
recovery_profiles = load_rich_profile_csv(profile_recovery_path, state_scope="recovery")

corr_pairs_df = load_correlation_pairs_csv(corr_pairs_normal_path)
group_map_df = load_group_map_csv(group_map_normal_path)
fault_pairings_df = load_fault_pairings_csv(fault_pairings_normal_path)

generator = SyntheticGenerator(
    normal_profiles=normal_profiles,
    abnormal_profiles=abnormal_profiles,
    recovery_profiles=recovery_profiles,
    correlation_pairs_dataframe=corr_pairs_df,
    group_map_dataframe=group_map_df,
    fault_pairings_dataframe=fault_pairings_df,
    random_seed=int(SYN_CFG["random_seed"]),
)

print("Sensors:", len(generator.sensors))
print("First sensors:", generator.sensors[:10])

logger.info("Generator Run")
logger.info("Generator Sensors Count: %s", len(generator.sensors))
logger.info("Generator Sensors List: %s", generator.sensors)

2026-03-14 01:39:52,522 | INFO | capstone.synthetic | Generator Run
2026-03-14 01:39:52,523 | INFO | capstone.synthetic | Generator Sensors Count: 50
2026-03-14 01:39:52,524 | INFO | capstone.synthetic | Generator Sensors List: ['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_51']


Sensors: 50
First sensors: ['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09']


In [11]:


print(inspect.signature(SyntheticGenerator.__init__))

logger.info("Generator Signature Inspection: %s", inspect.signature(SyntheticGenerator.__init__))

2026-03-14 01:39:52,537 | INFO | capstone.synthetic | Generator Signature Inspection: (self, *, normal_profiles: 'Dict[str, SensorRichProfile]', abnormal_profiles: 'Dict[str, SensorRichProfile]', recovery_profiles: 'Dict[str, SensorRichProfile]', correlation_pairs_dataframe: 'pd.DataFrame', group_map_dataframe: 'pd.DataFrame', fault_pairings_dataframe: 'pd.DataFrame', random_seed: 'int' = 42) -> 'None'


(self, *, normal_profiles: 'Dict[str, SensorRichProfile]', abnormal_profiles: 'Dict[str, SensorRichProfile]', recovery_profiles: 'Dict[str, SensorRichProfile]', correlation_pairs_dataframe: 'pd.DataFrame', group_map_dataframe: 'pd.DataFrame', fault_pairings_dataframe: 'pd.DataFrame', random_seed: 'int' = 42) -> 'None'


In [12]:
rng = np.random.default_rng(int(SYN_CFG.get("random_seed", 42)))

ep_cfg = SYN_CFG["episode"]

NORMAL_BEFORE = sample_int_range(rng, ep_cfg["normal_before_range"])
BUILDUP       = sample_int_range(rng, ep_cfg["buildup_range"])
FAILURE       = sample_int_range(rng, ep_cfg["failure_range"])
RECOVERY      = sample_int_range(rng, ep_cfg["recovery_range"])
NORMAL_AFTER  = sample_int_range(rng, ep_cfg["normal_after_range"])
MAGNITUDE     = sample_float_range(rng, ep_cfg["magnitude_range"])

sample_ranges_dict = {
    "normal_before_range": ep_cfg["normal_before_range"],
    "normal_before_selection": NORMAL_BEFORE,
    "buildup_range": ep_cfg["buildup_range"],
    "buildup_selection": BUILDUP,
    "failure_range": ep_cfg["failure_range"],
    "failure_selection": FAILURE,
    "recovery_range": ep_cfg["recovery_range"],
    "recovery_selection": RECOVERY,
    "normal_after_range": ep_cfg["normal_after_range"],
    "normal_after_selection": NORMAL_AFTER,
    "magnitude_range": ep_cfg["magnitude_range"],
    "magnitude_selection": MAGNITUDE
}

logger.info("Samples Selected From Episode Ranges: %s", sample_ranges_dict)



primary_sensor = PRIMARY_SENSOR or generator.sensors[0]

logger.info("Primary Sensor: %s", primary_sensor)



rng = np.random.default_rng(int(SYN_CFG.get("random_seed", 42)))

allowed_faults = SYN_CFG["faults"]["allowed"]

if PRIMARY_FAULT_TYPE is None or str(PRIMARY_FAULT_TYPE).strip() == "":
    primary_fault_type = str(rng.choice(allowed_faults))
else:
    # If someone accidentally passed a list, fix it here too
    if isinstance(PRIMARY_FAULT_TYPE, (list, tuple)):
        primary_fault_type = str(rng.choice(PRIMARY_FAULT_TYPE))
    else:
        primary_fault_type = str(PRIMARY_FAULT_TYPE)


logger.info("Allowed Fault Types, %s", allowed_faults)
logger.info("Primary Fault Selected, %s", primary_fault_type)


episode = EpisodeSpec(
    primary_sensor=primary_sensor,
    primary_fault_type=primary_fault_type,
    magnitude=MAGNITUDE,
    normal_before=NORMAL_BEFORE,
    buildup=BUILDUP,
    failure=FAILURE,
    recovery=RECOVERY,
    normal_after=NORMAL_AFTER,
)

synthetic_df = generator.generate_episode(episode)

print("Generated rows:", len(synthetic_df))
display(synthetic_df.head(10))
display(synthetic_df["stream_state"].value_counts())

logger.info("Episodes Generatored : %s", sample_ranges_dict)
logger.info("Generatored Rows: %s", len(synthetic_df))
logger.info("Stream State Value Counts : %s", synthetic_df["stream_state"].value_counts())


2026-03-14 01:39:52,561 | INFO | capstone.synthetic | Samples Selected From Episode Ranges: {'normal_before_range': [200, 600], 'normal_before_selection': 235, 'buildup_range': [50, 140], 'buildup_selection': 120, 'failure_range': [40, 120], 'failure_selection': 93, 'recovery_range': [80, 180], 'recovery_selection': 124, 'normal_after_range': [200, 600], 'normal_after_selection': 373, 'magnitude_range': [0.75, 2.0], 'magnitude_selection': 1.6217100363242047}
2026-03-14 01:39:52,565 | INFO | capstone.synthetic | Primary Sensor: sensor_00
2026-03-14 01:39:52,567 | INFO | capstone.synthetic | Allowed Fault Types, ['drift_up', 'drift_down', 'spike', 'stuck_constant', 'variance_burst', 'step_shift', 'intermittent_dropout', 'sawtooth']
2026-03-14 01:39:52,568 | INFO | capstone.synthetic | Primary Fault Selected, drift_up


Generated rows: 945


/workspace/utils/synthetic_generator.py:167: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe[sensor] = x


,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,...,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,stream_state,phase
0,2.352243,48.224049,51.756289,44.281130,611.651609,76.177824,13.269877,15.917599,15.244823,14.828713,...,43.341399,40.502652,44.691516,50.261989,42.200690,151.308390,53.665486,198.131414,normal,normal
1,2.376164,48.235667,51.784998,44.337527,616.328898,76.075726,13.370036,15.980635,15.304309,14.889547,...,42.261864,40.521069,43.730729,49.076013,42.257428,151.535714,53.801454,198.562431,normal,normal
2,2.419480,48.190663,51.804553,44.381537,624.785470,75.937147,13.523224,16.082557,15.401887,14.982816,...,42.481958,40.638697,42.389691,47.372806,42.517715,150.758235,53.798237,199.002846,normal,normal
3,2.453888,48.208080,51.528053,44.183434,632.271206,75.916633,13.641444,16.167481,15.480140,15.061733,...,42.648747,40.513016,41.213290,46.381180,42.613085,150.582602,53.777787,198.750930,normal,normal
4,2.465576,48.155154,51.611270,44.260033,636.236209,75.853356,13.697895,16.201576,15.516185,15.098538,...,42.888248,40.648393,41.147901,46.225968,42.797586,149.608452,53.987161,198.698516,normal,normal
5,2.457706,48.142508,51.589872,44.244213,634.468260,75.811571,13.676989,16.185569,15.496472,15.081289,...,42.888565,40.685704,41.130533,45.988517,42.846390,149.213812,53.789729,199.450364,normal,normal
6,2.483829,48.168305,51.468144,44.166747,639.635990,75.766506,13.759614,16.242508,15.556620,15.141471,...,43.816699,40.669946,40.587491,45.126040,42.849217,148.718601,53.938087,198.613272,normal,normal
7,2.461624,48.190666,51.498339,44.188093,635.031167,75.906472,13.687466,16.194224,15.505951,15.089802,...,43.457677,40.690986,42.053004,46.815928,43.048895,148.266187,54.053539,198.893700,normal,normal
8,2.453522,48.182119,51.723696,44.396789,633.923444,75.710039,13.661642,16.173887,15.492122,15.068781,...,42.895899,40.630947,41.395713,45.953568,43.030712,149.021335,54.088751,199.081592,normal,normal
9,2.426738,48.182421,51.823100,44.482695,628.263384,75.673409,13.578150,16.106140,15.432618,15.014121,...,42.062689,40.519340,42.374632,47.354480,43.024444,147.794300,53.950527,198.855046,normal,normal


stream_state
normal      608
recovery    124
buildup     120
abnormal     93
Name: count, dtype: int64

2026-03-14 01:39:52,753 | INFO | capstone.synthetic | Episodes Generatored : {'normal_before_range': [200, 600], 'normal_before_selection': 235, 'buildup_range': [50, 140], 'buildup_selection': 120, 'failure_range': [40, 120], 'failure_selection': 93, 'recovery_range': [80, 180], 'recovery_selection': 124, 'normal_after_range': [200, 600], 'normal_after_selection': 373, 'magnitude_range': [0.75, 2.0], 'magnitude_selection': 1.6217100363242047}
2026-03-14 01:39:52,755 | INFO | capstone.synthetic | Generatored Rows: 945
2026-03-14 01:39:52,756 | INFO | capstone.synthetic | Stream State Value Counts : stream_state
normal      608
recovery    124
buildup     120
abnormal     93
Name: count, dtype: int64


In [13]:
engine = get_engine(env_prefix="DB")

batch_seq = f"seq_synthetic_{DATASET_NAME}_batch_id"
cycle_seq = f"seq_synthetic_{DATASET_NAME}_cycle_id"

ensure_sequence(engine, schema=PG_SCHEMA, sequence_name=batch_seq)
ensure_sequence(engine, schema=PG_SCHEMA, sequence_name=cycle_seq)

batch_id = reserve_next_batch_id(engine, schema=PG_SCHEMA, sequence_name=batch_seq)
cycle_start = reserve_cycle_range(engine, schema=PG_SCHEMA, sequence_name=cycle_seq, n_rows=len(synthetic_df))

table_name = write_stream_batch(
    engine,
    synthetic_df,
    dataset_name=DATASET_NAME,
    schema=PG_SCHEMA,
    artifact_name=ARTIFACT_NAME,
    batch_id=batch_id,
    cycle_start=cycle_start,
)

print("Wrote to:", table_name)
print("batch_id:", batch_id, "cycle_start:", cycle_start)

Wrote to: synthetic_pump_stream
batch_id: 3 cycle_start: 1891


In [14]:
import os
{k: os.environ.get(k) for k in ["DB_HOST","DB_PORT","DB_NAME","DB_USER","DB_PASSWORD","POSTGRES_DB","POSTGRES_USER","POSTGRES_PASSWORD"]}

{'DB_HOST': 'dcook_capstone_postgres',
 'DB_PORT': '5432',
 'DB_NAME': 'dcook_capstone_postgres_db',
 'DB_USER': 'dcook_admin',
 'DB_PASSWORD': 'V9m!pQ4z@H2eS7wK',
 'POSTGRES_DB': None,
 'POSTGRES_USER': None,
 'POSTGRES_PASSWORD': None}

In [15]:
import importlib.util
print("psycopg:", importlib.util.find_spec("psycopg"))
print("psycopg2:", importlib.util.find_spec("psycopg2"))

psycopg: None
psycopg2: ModuleSpec(name='psycopg2', loader=<_frozen_importlib_external.SourceFileLoader object at 0x77036723d4b0>, origin='/opt/miniconda/envs/capstone/lib/python3.10/site-packages/psycopg2/__init__.py', submodule_search_locations=['/opt/miniconda/envs/capstone/lib/python3.10/site-packages/psycopg2'])


In [16]:
import os
from utils.postgres_util import build_postgres_url_from_env

os.environ["DB_DATABASE"] = os.environ["DB_NAME"]  # alias

url = build_postgres_url_from_env(prefix="DB", driver="psycopg2")
print("URL (redacted):", url.split("@")[0] + "@<HOST_REDACTED>")
print("Raw host env:", os.environ.get("DB_HOST"))
print("Raw user env:", os.environ.get("DB_USER"))
print("Raw db env:", os.environ.get("DB_DATABASE"))
print("Raw password has @?:", "@" in (os.environ.get("DB_PASSWORD") or ""))

URL (redacted): postgresql+psycopg2://dcook_admin:V9m%21pQ4z%40H2eS7wK@<HOST_REDACTED>
Raw host env: dcook_capstone_postgres
Raw user env: dcook_admin
Raw db env: dcook_capstone_postgres_db
Raw password has @?: True


In [17]:
process_run_id = make_process_run_id(str(SYN_CFG.get("process_run_id_prefix", "synthetic")))

synthetic_truth = initialize_layer_truth(
    truth_version=str(TRUTH_VERSION),
    dataset_name=DATASET_NAME,
    layer_name="synthetic",
    process_run_id=process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=PARENT_TRUTH_HASH,
)

LAYER_NAME = "synthetic"

resolved_config_dir = ARTIFACTS_ROOT / "synthetic" / DATASET_NAME
resolved_config_dir.mkdir(parents=True, exist_ok=True)

resolved_config_path = resolved_config_dir / f"{DATASET_NAME}__{LAYER_NAME}__resolved_config.yaml"

# export_config_snapshot requires a destination path
export_config_snapshot(CONFIG, destination=resolved_config_path)

print("Resolved config written to:", resolved_config_path)


synthetic_truth = update_truth_section(
    synthetic_truth,
    "config_snapshot",
    {
        # store the path you just wrote
        "resolved_config_path": str(resolved_config_path),

        # optional: small inline config view (JSON-safe)
        "truth_config_block": build_truth_config_block(CONFIG),

        # keep your synthetic config subset if you want
        "synthetic_cfg": SYN_CFG,
    },
)

synthetic_truth = update_truth_section(
    synthetic_truth,
    "runtime_facts",
    {
        "primary_sensor": primary_sensor,
        "primary_fault_type": primary_fault_type,
        "episode": episode.__dict__,
        "normal_before_range": ep_cfg["normal_before_range"],
        "normal_before_selection": NORMAL_BEFORE,
        "buildup_range": ep_cfg["buildup_range"],
        "buildup_selection": BUILDUP,
        "failure_range": ep_cfg["failure_range"],
        "failure_selection": FAILURE,
        "recovery_range": ep_cfg["recovery_range"],
        "recovery_selection": RECOVERY,
        "normal_after_range": ep_cfg["normal_after_range"],
        "normal_after_selection": NORMAL_AFTER,
        "magnitude_range": ep_cfg["magnitude_range"],
        "magnitude_selection": MAGNITUDE,
        "row_count": int(len(synthetic_df)),
        "parent_truth_hash": PARENT_TRUTH_HASH,
        "silver_eda_truth_hash": PARENT_TRUTH_HASH,
    },
)

# Optionally save local parquet artifact too (useful for debugging)
synth_dir = ARTIFACTS_ROOT / "synthetic" / DATASET_NAME
synth_dir.mkdir(parents=True, exist_ok=True)
out_path = synth_dir / f"{DATASET_NAME}__synthetic__episode.parquet"
save_data(synthetic_df, synth_dir, out_path.name)

artifact_paths_payload = {
    "silver_eda_truth_hash": PARENT_TRUTH_HASH,
    "profile_normal_path": profile_normal_path,
    "profile_abnormal_path": profile_abnormal_path,
    "profile_recovery_path": profile_recovery_path,
    "corr_pairs_normal_path": corr_pairs_normal_path,
    "group_map_normal_path": group_map_normal_path,
    "fault_pairings_normal_path": fault_pairings_normal_path,
    "synthetic_episode_path": str(out_path),
    "postgres_schema": PG_SCHEMA,
    "postgres_table": table_name,
    "postgres_batch_id": int(batch_id),
    "postgres_cycle_start": int(cycle_start),
}

if EXPORT_ENABLED:
    artifact_paths_payload["export_batch_parquet_path"] = str(export_path)

synthetic_truth = update_truth_section(synthetic_truth, "artifact_paths", artifact_paths_payload)

meta_columns = sorted(["meta__truth_hash", "meta__parent_truth_hash", "meta__pipeline_mode"])
feature_columns = sorted([c for c in synthetic_df.columns if not str(c).startswith("meta__")])

truth_record = build_truth_record(
    truth_base=synthetic_truth,
    row_count=int(len(synthetic_df)),
    column_count=int(synthetic_df.shape[1] + 3),
    meta_columns=meta_columns,
    feature_columns=feature_columns,
)

synth_truth_hash = truth_record["truth_hash"]

# stamp lineage columns into dataframe (optional)
synthetic_df = stamp_truth_columns(
    synthetic_df,
    truth_hash=synth_truth_hash,
    parent_truth_hash=PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

truth_path = save_truth_record(
    truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name="synthetic",
)

append_truth_index(truth_record, truth_index_path=TRUTH_INDEX_PATH)

print("Synthetic truth hash:", synth_truth_hash)
print("Synthetic truth path:", truth_path)
print("Local episode parquet:", out_path)

2026-03-14 01:39:54,183 | INFO | capstone.file_io | Saving DataFrame to Parquet: /workspace/artifacts/synthetic/pump/pump__synthetic__episode.parquet


Resolved config written to: /workspace/artifacts/synthetic/pump/pump__synthetic__resolved_config.yaml


2026-03-14 01:39:54,287 | INFO | capstone.file_io | Saved: pump__synthetic__episode.parquet | shape=(945, 52) | columns=['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_51', 'stream_state', 'phase']


NameError: name 'export_path' is not defined